<a href="https://colab.research.google.com/github/tezendrax/DDoS-Attack-Detection-using-Machine-Learning/blob/main/ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from scipy.io import arff

# Load the ARFF file
data, meta = arff.loadarff('/content/final-dataset (1).arff')

# Convert to pandas DataFrame
df = pd.DataFrame(data)

# Show basic information
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Take a random sample of 50,000 rows
df_sample = df.sample(n=50000, random_state=42)

# Reset index (important for clean data)
df_sample = df_sample.reset_index(drop=True)

# Check new shape
print("New dataset shape:", df_sample.shape)

# See first few rows
df_sample.head()

In [ ]:
selected_columns = [
    'PKT_SIZE',
    'PKT_RATE',
    'BYTE_RATE',
    'NUMBER_OF_PKT',
    'PKT_DELAY_NODE',
    'PKT_TYPE',
    'PKT_CLASS'   # target (VERY IMPORTANT)
]

In [ ]:
# Select only important columns
df_selected = df_sample[selected_columns]

# Check shape
print("Selected dataset shape:", df_selected.shape)

# View data
df_selected.head()

In [ ]:
# Select only important columns (with copy)
df_selected = df_sample[selected_columns].copy()

# Convert byte strings to normal strings
for col in df_selected.columns:
    if df_selected[col].dtype == 'object':
        df_selected.loc[:, col] = df_selected[col].apply(
            lambda x: x.decode('utf-8') if isinstance(x, bytes) else x
        )

In [ ]:
df_selected.head()

In [ ]:
# Create binary target column
df_selected['TARGET'] = df_selected['PKT_CLASS'].apply(lambda x: 0 if x == 'Normal' else 1)

# Check distribution
print(df_selected['TARGET'].value_counts())

# View updated data
df_selected.head()


#   We converted the multiclass target into a binary classification problem by
#   labeling normal traffic as 0 and attack traffic as 1 to simplify the
#   detection task.



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Step 5.1: Separate features and target
X = df_selected.drop(columns=['PKT_CLASS', 'TARGET'])
y = df_selected['TARGET']

# Step 5.2: Encode categorical column (PKT_TYPE)
encoder = LabelEncoder()
X['PKT_TYPE'] = encoder.fit_transform(X['PKT_TYPE'])

# Check result
print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()


#   We separated the input features and target variable, and encoded categorical
#   features like packet type into numerical values to make them compatible
#   with machine learning models.
# tcp ? 0
# ack ? 1
# cbr ? 2

In [ ]:
print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

In [ ]:
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

# 7) Create scaler
scaler = StandardScaler()

# Fit on training data ONLY and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

# Check shape
print("Scaled X_train:", X_train_scaled.shape)
print("Scaled X_test:", X_test_scaled.shape)

In [ ]:
# 8) Train Models

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# -------------------------
# Model 1: Logistic Regression
# -------------------------
log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

# -------------------------
# Model 2: Random Forest
# -------------------------
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)   # NOTE: RF doesn't need scaling

# -------------------------
# Model 3: Support Vector Machine (SVM)
# -------------------------
svm_model = SVC()
svm_model.fit(X_train_scaled, y_train)

print("All models trained successfully ")

In [ ]:
# =========================
# STEP 9: Evaluate Models
# =========================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# -------------------------
# Predictions
# -------------------------
y_pred_log = log_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test)
y_pred_svm = svm_model.predict(X_test_scaled)

# -------------------------
# Logistic Regression Results
# -------------------------
print("===== Logistic Regression =====")
print("Accuracy :", accuracy_score(y_test, y_pred_log))
print("Precision:", precision_score(y_test, y_pred_log))
print("Recall   :", recall_score(y_test, y_pred_log))
print("F1 Score :", f1_score(y_test, y_pred_log))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))
print("Classification Report:\n", classification_report(y_test, y_pred_log))

# -------------------------
# Random Forest Results
# -------------------------
print("\n===== Random Forest =====")
print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# -------------------------
# SVM Results
# -------------------------
print("\n===== SVM =====")
print("Accuracy :", accuracy_score(y_test, y_pred_svm))
print("Precision:", precision_score(y_test, y_pred_svm))
print("Recall   :", recall_score(y_test, y_pred_svm))
print("F1 Score :", f1_score(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("Classification Report:\n", classification_report(y_test, y_pred_svm))

In [ ]:
# =========================
# STEP 10: Compare Models and Final Conclusion
# =========================

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Create comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVM'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_svm)
    ],
    'Precision': [
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_svm)
    ],
    'Recall': [
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_svm)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_svm)
    ]
})

print(results)

In [ ]:
# =========================
# GRAPH 1: Model Comparison
# =========================

import matplotlib.pyplot as plt

# Plot bar chart
results.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1 Score']].plot(kind='bar')

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

In [ ]:
# =========================
# GRAPH 2: Confusion Matrix
# =========================

import seaborn as sns
from sklearn.metrics import confusion_matrix

# Use best model prediction (example: Random Forest)
cm = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# =========================
# GRAPH 3: Target Distribution
# =========================

import seaborn as sns

sns.countplot(x=y)
plt.title("Distribution of Target Classes")
plt.xlabel("Class (0 = Normal, 1 = Attack)")
plt.ylabel("Count")
plt.show()